# Connecting to Client

In [1]:
from ib_insync import *
util.startLoop()  # uncomment this line when in a notebook

ib = IB()
ib.connect('127.0.0.1', 7496, clientId=1)

# ib.connect('127.0.0.1', 4002, clientId=1) 
# use above for IB gateway


<IB connected to 127.0.0.1:7496 clientId=1>

# Install required libraries  - pre-installed int he venv kernel

In [2]:
from bs4 import BeautifulSoup as bs
import pandas as pd

In [3]:
stock = Stock ('NVDA', 'smart', 'USD')

bars = ib.reqHistoricalData(
    stock, endDateTime='', durationStr='30 D',
    barSizeSetting='1 hour', whatToShow='MIDPOINT', useRTH=True)

# convert to pandas dataframe (pandas needs to be installed):
df = util.df(bars)
print(df)

                         date    open    high     low   close  volume  \
0   2026-01-02 09:30:00-05:00  189.85  192.90  189.58  191.74    -1.0   
1   2026-01-02 10:00:00-05:00  191.74  192.24  188.82  188.96    -1.0   
2   2026-01-02 11:00:00-05:00  188.96  190.07  188.29  189.34    -1.0   
3   2026-01-02 12:00:00-05:00  189.34  189.94  189.07  189.46    -1.0   
4   2026-01-02 13:00:00-05:00  189.46  189.78  188.26  188.59    -1.0   
..                        ...     ...     ...     ...     ...     ...   
205 2026-02-13 11:00:00-05:00  183.31  184.17  182.79  183.84    -1.0   
206 2026-02-13 12:00:00-05:00  183.84  184.98  183.66  183.99    -1.0   
207 2026-02-13 13:00:00-05:00  183.99  184.86  183.46  184.20    -1.0   
208 2026-02-13 14:00:00-05:00  184.20  184.29  182.82  182.87    -1.0   
209 2026-02-13 15:00:00-05:00  182.87  183.42  181.58  182.79    -1.0   

     average  barCount  
0       -1.0        -1  
1       -1.0        -1  
2       -1.0        -1  
3       -1.0        -1 

In [6]:
def get_option_chain(symbol='SPY', exchange='SMART'):
    """
    Connects to IBKR and fetches the full option chain for a given stock.
    """
    ib = IB()
    try:
        # Connect to TWS or IB Gateway (commnted below line as ib is already connected by runing code in cell 1
        #ib.connect('127.0.0.1', 7496, clientId=1)

        # Define the underlying stock contract
        underlying_stock = Stock(symbol, exchange, 'USD')
        ib.qualifyContracts(underlying_stock)

        # Fetch option chain parameters (expirations and strikes)
        chains = ib.reqSecDefOptParams(underlying_stock.symbol, '', underlying_stock.secType, underlying_stock.conId)

        if not chains:
            print(f"No option chain found for {symbol}.")
            return pd.DataFrame()

        # Filter for a specific exchange if necessary, e.g., 'SMART'
        chain = next(c for c in chains if c.exchange == exchange)

        # Create all possible option contracts
        option_contracts = []
        for expiration in chain.expirations:
            for strike in chain.strikes:
                for right in ['C', 'P']: # C for Call, P for Put
                    contract = Option(symbol, expiration, strike, right, exchange, chain.multiplier, 'USD')
                    option_contracts.append(contract)
        
        # Qualify contracts to ensure they are valid
        qualified_contracts = ib.qualifyContracts(*option_contracts)
        print(f"Found {len(qualified_contracts)} valid option contracts.")

        if not qualified_contracts:
            return pd.DataFrame()

        # Request market data for all contracts
        tickers = ib.reqTickers(*qualified_contracts)

        # Format data into a DataFrame
        option_data = []
        for ticker in tickers:
            contract = ticker.contract
            option_data.append({
                'Symbol': contract.symbol,
                'Expiry': contract.lastTradeDateOrContractMonth,
                'Strike': contract.strike,
                'Type': contract.right,
                'Bid': ticker.bid,
                'Ask': ticker.ask,
                'Last': ticker.last,
                'Volume': ticker.volume,
                'OpenInterest': ticker.openInterest,
                'ImpliedVolatility': ticker.impliedVolatility,
                'Delta': ticker.delta,
                'Gamma': ticker.gamma,
                'Vega': ticker.vega,
                'Theta': ticker.theta
            })

        return pd.DataFrame(option_data)

    except Exception as e:
        print(f"An error occurred: {e}")
        return pd.DataFrame()
    finally:
        ib.disconnect()

if __name__ == '__main__':
    # Example: Get option chain for AAPL
    option_chain_df = get_option_chain(symbol='AAPL', exchange='SMART')
    
    if not option_chain_df.empty:
        print("AAPL Option Chain (first 10 rows):")
        print(option_chain_df.head(10))



An error occurred: Not connected


In [ ]:

def get_option_chain(symbol='SPY', exchange='SMART'):
    """
    Connects to IBKR and fetches the full option chain for a given stock.
    """
    #ib = IB()
    try:
        # Connect to TWS or IB Gateway
        #ib.connect('127.0.0.1', 7497, clientId=1)

        # Define the underlying stock contract
        underlying_stock = Stock(symbol, exchange, 'USD')
        ib.qualifyContracts(underlying_stock)

        # Fetch option chain parameters (expirations and strikes)
        chains = ib.reqSecDefOptParams(underlying_stock.symbol, '', underlying_stock.secType, underlying_stock.conId)

        if not chains:
            print(f"No option chain found for {symbol}.")
            return pd.DataFrame()

        # Filter for a specific exchange if necessary, e.g., 'SMART'
        chain = next(c for c in chains if c.exchange == exchange)

        # Create all possible option contracts
        option_contracts = []
        for expiration in chain.expirations:
            for strike in chain.strikes:
                for right in ['C', 'P']: # C for Call, P for Put
                    contract = Option(symbol, expiration, strike, right, exchange, chain.multiplier, 'USD')
                    option_contracts.append(contract)
        
        # Qualify contracts to ensure they are valid
        qualified_contracts = ib.qualifyContracts(*option_contracts)
        print(f"Found {len(qualified_contracts)} valid option contracts.")

        if not qualified_contracts:
            return pd.DataFrame()

        # Request market data for all contracts
        tickers = ib.reqTickers(*qualified_contracts)

        # Format data into a DataFrame
        option_data = []
        for ticker in tickers:
            contract = ticker.contract
            option_data.append({
                'Symbol': contract.symbol,
                'Expiry': contract.lastTradeDateOrContractMonth,
                'Strike': contract.strike,
                'Type': contract.right,
                'Bid': ticker.bid,
                'Ask': ticker.ask,
                'Last': ticker.last,
                'Volume': ticker.volume,
                'OpenInterest': ticker.openInterest,
                'ImpliedVolatility': ticker.impliedVolatility,
                'Delta': ticker.delta,
                'Gamma': ticker.gamma,
                'Vega': ticker.vega,
                'Theta': ticker.theta
            })

        return pd.DataFrame(option_data)

    except Exception as e:
        print(f"An error occurred: {e}")
        return pd.DataFrame()
    finally:
        ib.disconnect()

if __name__ == '__main__':
    # Example: Get option chain for AAPL
    option_chain_df = get_option_chain(symbol='AAPL', exchange='SMART')
    
    if not option_chain_df.empty:
        print("AAPL Option Chain (first 10 rows):")
        print(option_chain_df.head(10))


Unknown contract: Option(symbol='AAPL', lastTradeDateOrContractMonth='20251017', strike=5.0, right='C', multiplier='100', exchange='SMART', currency='USD')
Unknown contract: Option(symbol='AAPL', lastTradeDateOrContractMonth='20251017', strike=5.0, right='P', multiplier='100', exchange='SMART', currency='USD')
Unknown contract: Option(symbol='AAPL', lastTradeDateOrContractMonth='20251017', strike=10.0, right='C', multiplier='100', exchange='SMART', currency='USD')
Unknown contract: Option(symbol='AAPL', lastTradeDateOrContractMonth='20251017', strike=10.0, right='P', multiplier='100', exchange='SMART', currency='USD')
Unknown contract: Option(symbol='AAPL', lastTradeDateOrContractMonth='20251017', strike=15.0, right='C', multiplier='100', exchange='SMART', currency='USD')
Unknown contract: Option(symbol='AAPL', lastTradeDateOrContractMonth='20251017', strike=15.0, right='P', multiplier='100', exchange='SMART', currency='USD')
Unknown contract: Option(symbol='AAPL', lastTradeDateOrContr

Found 2364 valid option contracts.


In [12]:
# This cell assumes you have an existing, connected 'ib' object.
# For example:
# from ib_insync import *
# ib = IB()
# ib.connect('127.0.0.1', 7497, clientId=1)

if __name__ == '__main__':
# Call the function with your connected 'ib' object and the desired symbol
    NVDA_options_df = get_option_chain(symbol='NVDA', exchange='SMART')

# Display the results in the notebook
    if not NVDA_options_df.empty:
        print("NVDA Option Chain (first 10 rows):")
        display(NVDA_options_df.head(10))






An error occurred: Not connected


In [7]:
# In your execution cell (Cell 2)  code does nto work - check later

# Call the function specifying NASDAQ as the primary exchange
NVDA_options_df = get_option_chain(ib, symbol='NVDA', exchange='NASDAQ')

# Display the results
if not NVDA_options_df.empty:
    print("\nNVDA Option Chain (first 10 rows):")
    display(NVDA_options_df.head(10))

TypeError: get_option_chain() got multiple values for argument 'symbol'